# Notebook 5 — Matriz de Retornos Simples e Logarítmicos

Construção das matrizes de retornos diários a partir do painel de preços alinhado
(saída do Notebook 4). São geradas **duas** matrizes, por uma razão metodológica e não
de mera conveniência:

- **Retorno simples** $R_{i,t} = \dfrac{P_{i,t}}{P_{i,t-1}} - 1$ — possui *aditividade
  cross-section*: o retorno de uma carteira é a média ponderada dos retornos simples dos
  ativos, $R_{p,t} = \sum_i w_i\,R_{i,t}$. Por isso é o insumo correto para a **construção
  e o backtest das carteiras** e para os otimizadores (Notebooks 7–10).
- **Retorno logarítmico** $r_{i,t} = \ln\!\left(\dfrac{P_{i,t}}{P_{i,t-1}}\right) = \ln(1+R_{i,t})$
  — possui *aditividade temporal* (o retorno de $k$ dias é a soma dos log-retornos diários),
  é aproximadamente simétrico e constitui o padrão da **modelagem econométrica** (ADF, KPSS,
  GARCH) empregada no Notebook 11.

> **Nota metodológica (imputação por *forward fill*).** Como o Notebook 1 aplicou `ffill`
> residual sobre lacunas pontuais, dias com preço repetido produzem retorno exatamente igual
> a zero. Esses zeros são contabilizados de forma diagnóstica (§4) para que a parcela artificial
> seja documentada; dado o baixíssimo número de lacunas preenchidas, o impacto sobre a
> volatilidade estimada é desprezível.

> **Convenção sobre a taxa livre de risco.** CDI e SELIC **não** são transformados em retorno:
> já são taxas diárias e seguem adiante como $r_f$. O retorno em excesso $R_{i,t}-r_{f,t}$ é uma
> transformação derivada trivial, calculada no ponto de consumo (otimizador de Sharpe e Notebook
> de inferência), e por isso **não** é persistido como artefato próprio, evitando redundância e
> risco de dessincronização. O excesso é definido sobre o retorno **simples**, coerente com a
> natureza aritmética da taxa diária.

## 1. Configuração do ambiente e parâmetros

In [1]:
from pathlib import Path
import logging
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] - %(message)s")

# Caminhos relativos à raiz do projeto (mesma convenção dos Notebooks 1, 2 e 4)
parent_path = Path.cwd().parent.parent

INPUT_DIR_PAINEL = parent_path / "data" / "Painel_Dados"
OUTPUT_DIR       = parent_path / "data" / "Retornos"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Parâmetros metodológicos (concentrados para auditoria)
TRADING_DAYS = 252          # pregões/ano para anualização
PREFIXO_ACAO = "ACAO_"      # prefixo das colunas de ações no painel
COL_IBOV     = "IBOV_close"
COLS_RF      = ["CDI_diario", "SELIC_diario"]

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print(f"[OK] Entrada (painel):  {INPUT_DIR_PAINEL}")
print(f"[OK] Saída (retornos):  {OUTPUT_DIR}")
print(f"[OK] Pregões/ano:       {TRADING_DAYS}")

[OK] Entrada (painel):  c:\VSCodeWorkspace\TCC_Final\data\Painel_Dados
[OK] Saída (retornos):  c:\VSCodeWorkspace\TCC_Final\data\Retornos
[OK] Pregões/ano:       252


## 2. Carregamento do painel alinhado

Lê preferencialmente o `.parquet` (preserva tipos e índice); recai no `.csv` se necessário.
Em seguida separa as colunas em três grupos: ações (universo investível), IBOVESPA (benchmark)
e taxas livres de risco.

In [2]:
def _carregar_painel():
    cand_parquet = INPUT_DIR_PAINEL / "painel_alinhado.parquet"
    cand_csv     = INPUT_DIR_PAINEL / "painel_alinhado.csv"
    # também tenta cwd e uploads, para facilitar testes locais
    extras = [Path.cwd() / "painel_alinhado.parquet",
              Path.cwd() / "painel_alinhado.csv"]
    if cand_parquet.exists():
        df = pd.read_parquet(cand_parquet)
    elif cand_csv.exists():
        df = pd.read_csv(cand_csv, index_col=0, parse_dates=True)
    else:
        for p in extras:
            if p.exists():
                df = (pd.read_parquet(p) if p.suffix == ".parquet"
                      else pd.read_csv(p, index_col=0, parse_dates=True))
                break
        else:
            raise FileNotFoundError(
                "painel_alinhado.(parquet|csv) não encontrado em "
                f"{INPUT_DIR_PAINEL} nem no diretório atual.")
    df = df.sort_index()
    df.index = pd.to_datetime(df.index)
    df.index.name = "data"
    return df

painel = _carregar_painel()

cols_acoes = [c for c in painel.columns if c.startswith(PREFIXO_ACAO)]
cols_rf    = [c for c in COLS_RF if c in painel.columns]
assert COL_IBOV in painel.columns, f"Coluna {COL_IBOV} ausente no painel."

print(f"Painel: {painel.shape[0]} pregões × {painel.shape[1]} colunas "
      f"({painel.index.min().date()} → {painel.index.max().date()})")
print(f"  Ações (universo investível): {len(cols_acoes)}")
print(f"  Benchmark:                   {COL_IBOV}")
print(f"  Taxas livres de risco:       {cols_rf}")

Painel: 3967 pregões × 125 colunas (2010-01-04 → 2025-12-30)
  Ações (universo investível): 122
  Benchmark:                   IBOV_close
  Taxas livres de risco:       ['CDI_diario', 'SELIC_diario']


## 3. Cálculo das matrizes de retornos

A primeira observação de cada série é necessariamente `NaN` (não há pregão anterior) e é
removida. Como os preços já foram saneados no Notebook 1 (estritamente positivos, sem `NaN`),
não há risco de divisão por zero nem de $\ln(0)$.

In [3]:
precos_acoes = painel[cols_acoes]
precos_ibov  = painel[COL_IBOV]

# --- Retornos SIMPLES ---
ret_simples      = precos_acoes.pct_change().iloc[1:]
ibov_ret_simples = precos_ibov.pct_change().iloc[1:]

# --- Retornos LOGARÍTMICOS ---
ret_log      = np.log(precos_acoes).diff().iloc[1:]
ibov_ret_log = np.log(precos_ibov).diff().iloc[1:]

print(f"[+] Retornos simples: {ret_simples.shape[0]} pregões × {ret_simples.shape[1]} ativos")
print(f"[+] Retornos log:     {ret_log.shape[0]} pregões × {ret_log.shape[1]} ativos")
print(f"    Janela de retornos: {ret_simples.index.min().date()} → {ret_simples.index.max().date()}")
print(f"    (perdeu-se 1 pregão na borda inicial — esperado)")

[+] Retornos simples: 3966 pregões × 122 ativos
[+] Retornos log:     3966 pregões × 122 ativos
    Janela de retornos: 2010-01-05 → 2025-12-30
    (perdeu-se 1 pregão na borda inicial — esperado)


### 3.1 Alinhamento da taxa livre de risco ao calendário de retornos

O $r_f$ não é transformado, apenas realinhado ao mesmo índice das matrizes de retornos
(perde-se a primeira data, por consistência), permitindo o cálculo direto de excesso a jusante.

In [4]:
rf = painel[cols_rf].reindex(ret_simples.index) if cols_rf else None
if rf is not None:
    print(f"[+] Taxa(s) livre(s) de risco alinhada(s): {rf.shape[0]} pregões × {rf.shape[1]} série(s)")
    print(rf.head(3))
else:
    print("[aviso] Nenhuma coluna de rf encontrada no painel.")

[+] Taxa(s) livre(s) de risco alinhada(s): 3966 pregões × 2 série(s)
            CDI_diario  SELIC_diario
data                                
2010-01-05    0.000328      0.000329
2010-01-06    0.000328      0.000329
2010-01-07    0.000328      0.000329


## 4. Testes de integridade pós-cálculo

Bateria de verificações sobre as matrizes antes da persistência. Falha em qualquer
asserção interrompe o pipeline. Os itens diagnósticos (não-bloqueantes) documentam a parcela
de retornos nulos oriundos da imputação.

In [5]:
print("[+] Executando testes de integridade...\n")

# T1 — índices idênticos entre as duas matrizes e monotônicos crescentes
assert ret_simples.index.equals(ret_log.index), "Falha T1: índices divergentes entre simples e log."
assert ret_simples.index.is_monotonic_increasing, "Falha T1: índice não-crescente."
print("      [OK] T1 — Índices idênticos e monotônicos crescentes")

# T2 — ausência de NaN após remoção da primeira linha
assert ret_simples.isna().sum().sum() == 0, "Falha T2: NaN em retornos simples."
assert ret_log.isna().sum().sum() == 0,     "Falha T2: NaN em retornos log."
print("      [OK] T2 — Sem NaN nas matrizes")

# T3 — ausência de valores infinitos
assert np.isfinite(ret_simples.values).all(), "Falha T3: ±inf em retornos simples."
assert np.isfinite(ret_log.values).all(),     "Falha T3: ±inf em retornos log."
print("      [OK] T3 — Sem valores infinitos")

# T4 — consistência interna: log == ln(1 + simples)
assert np.allclose(ret_log.values, np.log1p(ret_simples.values), atol=1e-10), \
    "Falha T4: log-retorno incompatível com ln(1+simples)."
print("      [OK] T4 — Consistência log ≡ ln(1 + simples)")

# T5 — dimensões coerentes com o universo de ativos
assert ret_simples.shape[1] == len(cols_acoes), "Falha T5: nº de colunas ≠ nº de ativos."
assert ret_simples.shape == ret_log.shape,      "Falha T5: shapes divergentes."
print("      [OK] T5 — Dimensões coerentes")

print("\n[OK] Matrizes aprovadas em todos os testes de integridade.")

# --- Diagnósticos não-bloqueantes ---
pct_zero_total = (ret_simples == 0).values.mean() * 100
top_zero = (ret_simples == 0).mean().sort_values(ascending=False).head(5) * 100
print(f"\n[Diagnóstico] Retornos exatamente nulos (proxy de ffill): {pct_zero_total:.3f}% do total")
print("  5 ativos com maior fração de dias nulos (%):")
print(top_zero.round(3).to_string())

[+] Executando testes de integridade...

      [OK] T1 — Índices idênticos e monotônicos crescentes
      [OK] T2 — Sem NaN nas matrizes
      [OK] T3 — Sem valores infinitos
      [OK] T4 — Consistência log ≡ ln(1 + simples)
      [OK] T5 — Dimensões coerentes

[OK] Matrizes aprovadas em todos os testes de integridade.

[Diagnóstico] Retornos exatamente nulos (proxy de ffill): 5.064% do total
  5 ativos com maior fração de dias nulos (%):
ACAO_FICT3   24.332000
ACAO_RPMG3   20.802000
ACAO_SANB4   18.306000
ACAO_HAGA4   17.852000
ACAO_WHRL4   17.574000


## 5. Verificação descritiva (*sanity check* estatístico)

Momentos diários e anualizados por ativo. O retorno médio anualizado usa o log-retorno
(aditivo no tempo, $\bar{r}\times 252$); a volatilidade anualizada usa
$\sigma_d\sqrt{252}$. Valores extremos ou implausíveis aqui sinalizariam erro a montante.

In [6]:
desc = pd.DataFrame({
    "Ret. médio diário":      ret_simples.mean(),
    "Vol. diária":            ret_simples.std(),
    "Ret. a.a. (log)":        ret_log.mean() * TRADING_DAYS,
    "Vol. a.a.":              ret_simples.std() * np.sqrt(TRADING_DAYS),
    "Assimetria":             ret_simples.skew(),
    "Curtose exc.":           ret_simples.kurtosis(),
    "Mín diário":             ret_simples.min(),
    "Máx diário":             ret_simples.max(),
})
desc.index = [c.replace(PREFIXO_ACAO, "") for c in desc.index]

print("Resumo dos momentos entre os ativos:")
print(desc.describe().loc[["mean", "min", "max"]].T.round(4).to_string())

print("\nBenchmark (IBOVESPA):")
print(f"  Ret. a.a. (log): {ibov_ret_log.mean()*TRADING_DAYS:.4f}  |  "
      f"Vol. a.a.: {ibov_ret_simples.std()*np.sqrt(TRADING_DAYS):.4f}")

desc.sort_values("Vol. a.a.", ascending=False).head(10).round(4)

Resumo dos momentos entre os ativos:
                       mean       min          max
Ret. médio diário  0.000600 -0.000600     0.004700
Vol. diária        0.029100  0.014600     0.300700
Ret. a.a. (log)    0.046100 -0.340000     0.221100
Vol. a.a.          0.462500  0.232000     4.773500
Assimetria         1.094800 -0.823200    58.579000
Curtose exc.      44.244200  1.857100 3,590.072900
Mín diário        -0.220900 -0.700000    -0.096300
Máx diário         0.424200  0.083900    18.470400

Benchmark (IBOVESPA):
  Ret. a.a. (log): 0.0529  |  Vol. a.a.: 0.2309


,Ret. médio diário,Vol. diária,Ret. a.a. (log),Vol. a.a.,Assimetria,Curtose exc.,Mín diário,Máx diário
GOLL54,0.004700,0.300700,-0.252700,4.773500,58.579000,"3,590.072900",-0.700000,18.470400
FICT3,0.003000,0.079400,0.061000,1.260400,4.431500,59.148600,-0.400000,1.422300
TELB4,0.001000,0.066200,-0.136600,1.051000,14.963200,432.159400,-0.343200,2.232100
INEP4,0.000300,0.050200,-0.213700,0.796500,2.335700,23.161500,-0.278500,0.712700
RPMG3,0.001000,0.048100,-0.020700,0.762900,3.449500,60.235700,-0.678600,0.804700
FHER3,0.000700,0.045200,-0.059400,0.717500,3.446100,59.568700,-0.433400,0.914400
RSID3,-0.000500,0.043000,-0.340000,0.682500,1.918600,18.585400,-0.236600,0.590900
PDTC3,-0.000200,0.039300,-0.230400,0.624600,0.981900,8.764700,-0.240800,0.369900
TASA4,0.000300,0.038100,-0.097000,0.605400,1.811800,24.307000,-0.386500,0.476500
PMAM3,-0.000600,0.037100,-0.306800,0.588900,1.805700,21.285400,-0.250000,0.550000


## 6. Persistência dos artefatos

Salva as matrizes de ações (simples e log) em `.parquet` e `.csv`, a série de retornos do
benchmark e a taxa livre de risco alinhada. Formato de data ISO e ponto decimal, eliminando
ambiguidade de localidade no carregamento posterior (Notebooks 6+).

In [7]:
def _salvar(df, nome, com_parquet=True):
    csv_path = OUTPUT_DIR / f"{nome}.csv"
    df.to_csv(csv_path, index=True, date_format="%Y-%m-%d", float_format="%.8f")
    msg = f"  {nome}.csv  ({csv_path.stat().st_size/1024:.1f} KB)"
    if com_parquet:
        pq_path = OUTPUT_DIR / f"{nome}.parquet"
        try:
            df.to_parquet(pq_path, engine="pyarrow")
            msg += f"  |  {nome}.parquet  ({pq_path.stat().st_size/1024:.1f} KB)"
        except Exception as e:
            msg += f"  |  [parquet falhou: {e}]"
    print(msg)

print("[+] Gravando artefatos em:", OUTPUT_DIR, "\n")

_salvar(ret_simples, "retornos_simples")
_salvar(ret_log,     "retornos_log")

ibov_ret = pd.DataFrame({"ibov_ret_simples": ibov_ret_simples,
                         "ibov_ret_log":     ibov_ret_log})
_salvar(ibov_ret, "ibov_retornos")

if rf is not None:
    _salvar(rf.rename(columns=str.lower), "rf_diario")

print("\n" + "=" * 60)
print("NOTEBOOK 5 CONCLUÍDO")
print("=" * 60)
print(f"Matriz de retornos: {ret_simples.shape[0]} pregões × {ret_simples.shape[1]} ativos")
print(f"Período: {ret_simples.index.min().date()} → {ret_simples.index.max().date()}")
print("Pronto para o cálculo de momentos/covariância e otimização (Notebook 6+).")
print("=" * 60)

[+] Gravando artefatos em: c:\VSCodeWorkspace\TCC_Final\data\Retornos 

  retornos_simples.csv  (5471.4 KB)  |  retornos_simples.parquet  (4293.5 KB)
  retornos_log.csv  (5471.4 KB)  |  retornos_log.parquet  (4301.8 KB)
  ibov_retornos.csv  (135.5 KB)  |  ibov_retornos.parquet  (109.3 KB)
  rf_diario.csv  (131.7 KB)  |  rf_diario.parquet  (39.3 KB)

NOTEBOOK 5 CONCLUÍDO
Matriz de retornos: 3966 pregões × 122 ativos
Período: 2010-01-05 → 2025-12-30
Pronto para o cálculo de momentos/covariância e otimização (Notebook 6+).


## 7. Apêndice sugerido para o TCC — texto de redação

> *"A partir do painel de preços alinhado, computaram-se as matrizes de retornos diários em
> duas formas complementares. O retorno simples, $R_{i,t} = P_{i,t}/P_{i,t-1} - 1$, foi adotado
> na construção e na avaliação das carteiras, por preservar a aditividade entre ativos
> ($R_{p,t} = \sum_i w_i R_{i,t}$). O retorno logarítmico, $r_{i,t} = \ln(1+R_{i,t})$, foi
> reservado à modelagem econométrica, por sua aditividade temporal e melhor aderência às
> hipóteses distribucionais dos testes empregados. A primeira observação de cada série, sem
> pregão anterior de referência, foi descartada. As séries de CDI e SELIC, por já constituírem
> taxas diárias, foram mantidas em nível e empregadas como taxa livre de risco; o retorno em
> excesso foi obtido por subtração no ponto de uso, sobre o retorno simples. A integridade das
> matrizes foi verificada quanto à ausência de valores ausentes ou infinitos, à coerência
> dimensional e à identidade $r_{i,t} \equiv \ln(1+R_{i,t})$, condição necessária para a
> consistência interna entre as duas representações."*

Os arquivos resultantes (`retornos_simples`, `retornos_log`, `ibov_retornos`, `rf_diario`)
são gravados em `data/Retornos/` em formato CSV e Parquet.